In [ ]:
import os
import time
import requests
import shutil
import zipfile
import io
import subprocess
import sys
import re
from pathlib import Path

# КОНФИГУРАЦИЯ
os.environ["NO_PROXY"] = "127.0.0.1,localhost,::1"
http = requests.Session()
http.trust_env = False
http.proxies = {"http": None, "https": None}

KOHARU_BASE = "http://127.0.0.1:4000/api/v1"
UPS_BIN = Path("/Applications/Upscayl.app/Contents/Resources/bin/upscayl-bin")
UPS_MODELS = Path("/Applications/Upscayl.app/Contents/Resources/models")
MODEL_NAME = "digital-art-4x"

PIPELINE_STEPS = ["pp-doclayout-v3", "comic-text-detector-seg", "speech-bubble-segmentation", "yuzumarker-font-detection", "paddle-ocr-vl-1.5", "llm", "lama-manga", "koharu-renderer"]
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.JPG', '.PNG', '.JPEG'}


def upscale_manga(path):
    """
    Универсальный апскейлер:
    - Если в папке есть картинки -> апскейлит как главу.
    - Если в папке только другие папки -> проходит по каждой и апскейлит их.
    """
    root_path = Path(path)
    if not root_path.exists():
        print(f"❌ Путь не найден: {root_path}")
        return

    # Проверяем, есть ли картинки непосредственно в этой папке
    images = [f for f in root_path.iterdir() if f.suffix.lower() in IMAGE_EXTS]
    
    if images:
        # Это папка с главой (в ней лежат картинки)
        _process_single_chapter(root_path, images)
    else:
        # Это корневая папка манги (в ней лежат папки глав)
        chapters = sorted([d for d in root_path.iterdir() if d.is_dir() and not d.name.startswith("temp_")])
        if not chapters:
            print(f"📁 В {root_path.name} не найдено ни глав, ни изображений.")
            return
        
        print(f"📚 Найдено глав для обработки: {len(chapters)}")
        for chap in chapters:
            chap_images = [f for f in chap.iterdir() if f.suffix.lower() in IMAGE_EXTS]
            if chap_images:
                _process_single_chapter(chap, chap_images)


def _process_single_chapter(chapter_path, images):
    """Внутренняя рабочая функция для одной главы (не вызывай её напрямую)"""
    total_files = len(images)
    print(f"\n🚀 Upscayl: {chapter_path.name} ({total_files} стр.)")

    temp_upscale = chapter_path.parent / f"temp_up_{int(time.time())}_{chapter_path.stem[-5:]}"
    temp_upscale.mkdir(exist_ok=True)

    try:
        cmd = [
            str(UPS_BIN), "-i", str(chapter_path.resolve()), "-o", str(temp_upscale.resolve()),
            "-m", str(UPS_MODELS.resolve()), "-n", MODEL_NAME, 
            "-z", "4", "-s", "2", "-f", "png", "-g", "0"
        ]

        process = subprocess.Popen(cmd, stderr=subprocess.PIPE, text=True, 
                                   bufsize=1, universal_newlines=True, cwd=str(UPS_BIN.parent))

        current_file_idx = 0
        last_line_was_zero = False

        for line in process.stderr:
            if "0.00%" in line and not last_line_was_zero:
                current_file_idx += 1
                last_line_was_zero = True
            if "%" in line:
                progress = line.strip().split('%')[0].split()[-1]
                display_idx = min(current_file_idx, total_files)
                print(f"   📈 Улучшение [{display_idx}/{total_files}]: {progress}%   ", end="\r")
                if "100.00%" in line: last_line_was_zero = False

        process.wait()

        # Замена оригиналов
        print(f"\n🔄 Замена файлов в {chapter_path.name}...")
        upscaled_files = list(temp_upscale.glob("*.png"))
        for up_file in upscaled_files:
            # Ищем оригинал по имени (stem)
            original_match = next((img for img in images if img.stem == up_file.stem), None)
            if original_match:
                original_match.unlink(missing_ok=True)
                shutil.move(str(up_file), str(chapter_path / up_file.name))
        
        print(f"✅ Готово: {chapter_path.name}")

    except Exception as e:
        print(f"❌ Ошибка в {chapter_path.name}: {e}")
    finally:
        if temp_upscale.exists():
            shutil.rmtree(temp_upscale)



def wait_and_grab_zip(page_ids, temp_dir):
    print(f"⏳ Мониторинг Геммы ({len(page_ids)} стр.)...")
    start_time = time.time()
    attempt = 1
    while True:
        try:
            payload = {"format": "rendered", "pages": page_ids}
            res = http.post(f"{KOHARU_BASE}/projects/current/export", json=payload, timeout=60)
            if res.status_code == 200 and 'zip' in res.headers.get('Content-Type', ''):
                with zipfile.ZipFile(io.BytesIO(res.content)) as z:
                    if len(z.namelist()) >= len(page_ids):
                        print(f"\n✅ ZIP получен за {int(time.time() - start_time)}с.")
                        z.extractall(temp_dir)
                        return True
            print(f"⌛ Попытка {attempt}: Рендер слоев... ({int(time.time() - start_time)}с)    ", end="\r")
        except: pass
        if time.time() - start_time > 1800: return False
        attempt += 1
        time.sleep(15)

def process_manga(MANGA_ROOT):
    # 1. Проверяем, есть ли изображения прямо в корневой папке
    root_images = sorted([f for f in MANGA_ROOT.iterdir() if f.suffix.lower() in IMAGE_EXTS])
    
    if root_images:
        # Если картинки есть, считаем, что это ОДНА глава
        print(f"📦 Режим одной главы: обнаружено {len(root_images)} изображений в корне.")
        chapters = [MANGA_ROOT]
    else:
        # Если картинок нет, ищем подпапки (старая логика)
        chapters = sorted([d for d in MANGA_ROOT.iterdir() if d.is_dir() and not d.name.startswith("temp_")])
        if not chapters:
            print("❌ Ошибка: В папке не найдено ни картинок, ни подпапок с главами.")
            return

    for chapter in chapters:
        # Получаем список картинок для текущей итерации
        images = sorted([f for f in chapter.iterdir() if f.suffix.lower() in IMAGE_EXTS])
        if not images: 
            continue

        print("\n" + "="*50)
        print(f"📂 ОБРАБОТКА: {chapter.name}")
        
        # Очистка имени только для Koharu (API)
        clean_name = re.sub(r'[^a-zA-Z0-9]', '', chapter.name)[:25]
        if not clean_name: clean_name = "DefaultName" # На случай, если имя папки только из символов
        
        res = http.post(f"{KOHARU_BASE}/projects", json={"name": f"Auto_{clean_name}"})
        project_id = res.json().get('id') if res.status_code == 200 else None
        if not project_id: 
            print(f"❌ Не удалось создать проект для {chapter.name}")
            continue
            
        http.put(f"{KOHARU_BASE}/projects/current", json={"id": project_id})
        
        id_to_name, page_ids = {}, []
        for img in images:
            with open(img, 'rb') as f:
                r = http.post(f"{KOHARU_BASE}/pages", files={'file': (img.name, f, 'image/jpeg')})
                if r.status_code == 200:
                    data = r.json()
                    p_list = data.get('pages', [])
                    if p_list:
                        last = p_list[-1]
                        pid = last if isinstance(last, str) else last.get('id')
                        page_ids.append(pid); id_to_name[pid] = img.name
                        print(f"📥 Загружено: {img.name}...", end="\r")
        
        print(f"\n🚀 Запуск нейросети...")
        http.post(f"{KOHARU_BASE}/pipelines", json={"steps": PIPELINE_STEPS, "pages": page_ids, "targetLanguage": "ru-RU"})
        
        # Временные папки создаем ВНУТРИ родительской директории (чтобы не мусорить внутри главы)
        temp_parent = chapter.parent if root_images else MANGA_ROOT
        temp_trans = temp_parent / f"temp_trans_{clean_name}"
        temp_upscale = temp_parent / f"temp_upscale_{clean_name}"
        
        shutil.rmtree(temp_trans, ignore_errors=True); shutil.rmtree(temp_upscale, ignore_errors=True)
        temp_trans.mkdir(exist_ok=True); temp_upscale.mkdir(exist_ok=True)

        if wait_and_grab_zip(page_ids, temp_trans):
            total_files = len(page_ids)
            print(f"🚀 Upscayl (M1 Pro)")
            try:
                cmd = [
                    str(UPS_BIN), "-i", str(temp_trans.resolve()), "-o", str(temp_upscale.resolve()),
                    "-m", str(UPS_MODELS.resolve()), "-n", MODEL_NAME, "-z", "4", "-s", "2", "-f", "png", "-g", "0"
                ]
                process = subprocess.Popen(cmd, stderr=subprocess.PIPE, text=True, bufsize=1, universal_newlines=True, cwd=str(UPS_BIN.parent))

                current_file_idx = 0
                last_line_was_zero = False
                last_printed_progress = ""

                for line in process.stderr:
                    if "0.00%" in line and not last_line_was_zero:
                        current_file_idx += 1
                        last_line_was_zero = True
                    
                    if "%" in line:
                        try:
                            progress = line.strip().split('%')[0].split()[-1]
                            current_log = f"    📈 Улучшение ({current_file_idx}/{total_files}): {progress}%    "
                            if current_log != last_printed_progress:
                                print(current_log, end="\r")
                                last_printed_progress = current_log
                        except: pass
                        
                        if "100.00%" in line:
                            last_line_was_zero = False
                
                process.wait()
                print("\n🔄 Замена оригиналов...")
                extracted = list(temp_upscale.glob("*.*"))
                for pid, orig_name in id_to_name.items():
                    match = next((f for f in extracted if pid in f.name), None)
                    if match and match.stat().st_size > 50000:
                        (chapter / orig_name).unlink(missing_ok=True)
                        shutil.move(str(match), str(chapter / f"{Path(orig_name).stem}.png"))
                
                print(f"✨ Готово: {chapter.name}")
            except Exception as e:
                print(f"❌ Ошибка в Upscayl: {e}")
        else:
            print(f"❌ ПРОПУСК: Сервер не отдал ZIP для {chapter.name}.")

        shutil.rmtree(temp_trans, ignore_errors=True); shutil.rmtree(temp_upscale, ignore_errors=True)

In [ ]:
MANGA_ROOT = Path("путь к папке с мангой")
process_manga(MANGA_ROOT)

📦 Режим одной главы: обнаружено 24 изображений в корне.

📂 ОБРАБОТКА: Schale scans_Ch.136_f81f8b
📥 Загружено: 025.png...
🚀 Запуск нейросети...
⏳ Мониторинг Геммы (24 стр.)...
⌛ Попытка 44: Рендер слоев... (678с)    
✅ ZIP получен за 695с.
🚀 Upscayl (M1 Pro)
    📈 Улучшение (1/24): 98.57%    
🔄 Замена оригиналов...
✨ Готово: Schale scans_Ch.136_f81f8b
